# Challenge 11: Solar Flare Classification: SVM Overfitting and Regularisation

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
   - [4.4 Preprocessing](#44-preprocessing)
5. [Part 1: The Overfitting SVM](#5-part-1-the-overfitting-svm)
   - [5.1 What Overfitting Looks Like](#51-what-overfitting-looks-like)
   - [5.2 Demonstrating Overfitting](#52-demonstrating-overfitting)
6. [Part 2: The Role of C and Gamma](#6-part-2-the-role-of-c-and-gamma)
   - [6.1 The C Parameter](#61-the-c-parameter)
   - [6.2 The Gamma Parameter](#62-the-gamma-parameter)
   - [6.3 The Joint Effect of C and Gamma](#63-the-joint-effect-of-c-and-gamma)
7. [Part 3: Finding the Right Regularisation](#7-part-3-finding-the-right-regularisation)
   - [7.1 Grid Search with Cross-Validation](#71-grid-search-with-cross-validation)
   - [7.2 Final Evaluation and Comparison](#72-final-evaluation-and-comparison)
   - [7.3 Reflection Questions](#73-reflection-questions)
8. [Solution](#8-solution)
9. [References](#9-references)

---
## 1. Introduction

Space weather forecasting is one of the most operationally demanding
real-time classification problems in science. NOAA's Space Weather
Prediction Center issues alerts around the clock based on data from
the GOES satellite network, NASA's Solar Dynamics Observatory, and
ground-based observatories worldwide. An X-class solar flare can
cause HF radio blackouts on the sunlit hemisphere within minutes of
peak emission; a coronal mass ejection (CME) arriving at Earth 1-3
days later can trigger geomagnetic storms that disrupt power grids,
GPS signals, and satellite operations. Automated classification of
solar activity from instrument measurements is therefore a genuinely
safety-critical task.

The **SolarObs** dataset contains 10,000 snapshots of solar observatory
measurements, each classified into one of three activity states:

- **Quiet** (~44%): background coronal emission, no significant flares
  or eruptions. Minimal space weather impact.
- **Active** (~36%): one or more active regions present, C or M-class
  flares possible. Elevated solar wind. Moderate space weather impact.
- **Eruptive** (~20%): X-class flare or CME in progress. Severe space
  weather. Immediate alerts required.

The dataset has 12 features: six informative spectroscopic and magnetic
measurements, and six noisy derived spectral ratio features that add
dimensionality without proportionate class signal. This combination
creates the conditions for one of the most common and instructive
failure modes in machine learning: a support vector machine that is
too flexible memorises the training data perfectly but generalises
poorly to unseen observations.

This challenge takes you through the full arc from overfitting
diagnosis to regularisation. You will see exactly how the SVM's two
key hyperparameters, `C` and `gamma`, interact to control the
bias-variance tradeoff, and you will use cross-validated grid search
to find the configuration that actually generalises.

### Learning Objectives

- Diagnose SVM overfitting by comparing training and test accuracy
  across a range of C and gamma values
- Understand the role of `C` (the regularisation inverse) in
  controlling the margin width and the number of support vectors
- Understand the role of `gamma` in the RBF kernel and why large
  gamma values produce overfitting regardless of C
- Produce and interpret a heatmap of training accuracy, test accuracy,
  and train-test gap across a C-gamma grid
- Use `GridSearchCV` with stratified cross-validation to find the
  jointly optimal (C, gamma) configuration
- Compare the overfitting model against the well-regularised model
  using the confusion matrix and per-class metrics

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `SVC` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html |
| SVM user guide and kernel tricks | https://scikit-learn.org/stable/modules/svm.html |
| `GridSearchCV` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html |
| RBF kernel and gamma explained | https://scikit-learn.org/stable/auto_examples/svm/plot_rbf_parameters.html |
| NOAA Space Weather Prediction Center | https://www.swpc.noaa.gov |
| NASA Solar Dynamics Observatory | https://sdo.gsfc.nasa.gov |
| GOES solar X-ray flux data | https://www.ngdc.noaa.gov/stp/satellite/goes/index.html |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook requires Python 3.9 or later and scikit-learn 1.0 or later.
The C-gamma grid sweep in Section 6.3 trains many SVMs; with `n_jobs=-1`
in GridSearchCV it will run in parallel and complete in 5-15 minutes
depending on hardware. The individual C and gamma sweeps in Sections 6.1
and 6.2 each take 1-3 minutes.

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations |
| `pandas` | Loading and inspecting data |
| `matplotlib` | Plotting (sweep curves, heatmaps, confusion matrices) |
| `seaborn` | Correlation heatmap and distribution plots |
| `sklearn.preprocessing` | `StandardScaler` |
| `sklearn.svm` | `SVC` |
| `sklearn.model_selection` | `train_test_split`, `GridSearchCV`, `StratifiedKFold` |
| `sklearn.metrics` | `classification_report`, `accuracy_score`, `ConfusionMatrixDisplay` |

### 3.3 Data

| Property | Value |
|---|---|
| Filename | `sol.csv` |
| Location | `data/sol.csv` (relative to this notebook) |
| Total rows | 10,000 |
| Features | 12 (6 informative + 6 noisy derived) |
| Target column | `activity_class` |
| Class 0 | quiet: ~4,395 samples (~44.0%) |
| Class 1 | active: ~3,608 samples (~36.1%) |
| Class 2 | eruptive: ~1,997 samples (~20.0%) |

The class distribution is moderately imbalanced: the eruptive class
represents rare but high-impact events, while quiet conditions are
the most common baseline state.

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

CLASS_NAMES  = ['quiet', 'active', 'eruptive']
FEATURE_COLS = [
    'log_xray_flux', 'halpha_intensity', 'magnetic_bmax', 'sunspot_area',
    'prominence_height', 'radio_burst_sfu',
    'uv_emission_index', 'temp_proxy_mk', 'solar_wind_speed',
    'spectral_ratio_1', 'spectral_ratio_2', 'spectral_ratio_3',
]
INFORMATIVE_COLS = FEATURE_COLS[:6]
NOISY_COLS       = FEATURE_COLS[6:]

print('Libraries loaded successfully.')
print(f'Informative features: {INFORMATIVE_COLS}')
print(f'Noisy derived features: {NOISY_COLS}')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/sol.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(8)

### 4.2 Understanding the Features

**Informative features** (measured directly from solar instruments):

| Feature | Units | Description |
|---|---|---|
| `log_xray_flux` | log10(W/m²) | Integrated 1-8 Å X-ray flux from GOES. The primary flare intensity indicator. Quiet: ~-8 to -6. Eruptive: ~-5 to -3. |
| `halpha_intensity` | relative | H-alpha chromospheric emission. Enhanced during flares. Quiet: ~1. Eruptive: 5-30. |
| `magnetic_bmax` | Gauss | Peak magnetic field in active region from SDO/HMI magnetogram. Quiet: <100 G. Eruptive: >500 G. |
| `sunspot_area` | 10⁻⁶ solar disk | Sunspot group area, proxy for magnetic flux. Quiet: <50. Eruptive: >300. |
| `prominence_height` | Mm | Height of tallest prominence above the solar limb. Eruptive prominences rise high before detaching. |
| `radio_burst_sfu` | solar flux units | Type II/III radio burst flux density. Quiet: <10 SFU. Eruptive: 100-10,000 SFU. |

**Noisy derived features** (computed from instrument combinations):

| Feature | Description |
|---|---|
| `uv_emission_index` | AIA 171/304 channel ratio plus calibration noise. Correlated with `log_xray_flux`. |
| `temp_proxy_mk` | Coronal temperature proxy from filter ratio [MK]. Correlated with `log_xray_flux`, noisy. |
| `solar_wind_speed` | Solar wind speed proxy [km/s]. Lags flare events by hours. Noisy. |
| `spectral_ratio_1` | Fe XII/Fe IX line ratio. Correlated with `temp_proxy_mk`. High measurement scatter. |
| `spectral_ratio_2` | Ca II/H-alpha ratio. Correlated with `halpha_intensity`. Noisy. |
| `spectral_ratio_3` | Third derived ratio. Near-pure noise for this classification task. |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
print('Class distribution:')
counts = df['activity_class'].value_counts().sort_index()
for k, v in counts.items():
    print(f'  Class {k} ({CLASS_NAMES[k]:10s}): {v:5d}  ({100*v/len(df):.1f}%)')

print()
print('Informative feature means by class:')
print(df.groupby('activity_class')[INFORMATIVE_COLS].mean().round(3).to_string())

**Questions to consider:**

- The three classes have clearly different means for `log_xray_flux`,
  `halpha_intensity`, and `sunspot_area`. Would a linear classifier
  do well on these features alone?
- The active class spans a wide range (C/M-class events cover two
  orders of magnitude in X-ray flux). This makes its boundary with
  both quiet and eruptive ambiguous. Where do you expect the SVM to
  make most of its errors?

In [ ]:
# Listing 4.
# Feature distributions for the informative features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colours = {0: 'steelblue', 1: 'darkorange', 2: 'firebrick'}

for ax, feat in zip(axes, INFORMATIVE_COLS):
    for cls in range(3):
        subset = df.loc[df['activity_class'] == cls, feat].dropna()
        subset.plot(kind='kde', ax=ax, label=CLASS_NAMES[cls],
                    color=colours[cls], linewidth=2)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=9)

plt.suptitle('Informative Feature Distributions by Solar Activity Class', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Questions to consider:**

- The quiet and active distributions overlap in `log_xray_flux` around
  -7 to -6.5. Similarly, active and eruptive overlap around -5 to -4.
  These boundary regions are where the SVM's decision boundary matters most.
- A high-C SVM will try to draw a tight boundary through these overlap
  regions, fitting every training example correctly regardless of whether
  the boundary generalises. What do you expect the test accuracy to be
  for a SVM that has perfectly memorised the training set?

In [ ]:
# Listing 5.
# Correlation matrix
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.4, ax=ax, square=True,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix (SolarObs)', fontsize=12)
# Mark boundary between informative and noisy features
ax.axhline(6, color='black', linewidth=2)
ax.axvline(6, color='black', linewidth=2)
ax.text(3, 6.3, 'Informative', ha='center', fontsize=9, fontweight='bold')
ax.text(9, 6.3, 'Noisy derived', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

**Questions to consider:**

- The noisy features are correlated with their parent informative features
  (e.g. `uv_emission_index` with `log_xray_flux`). A high-C SVM can exploit
  these correlations to find spurious boundaries in the noisy dimensions that
  perfectly separate training examples but do not generalise.
- `spectral_ratio_3` is nearly uncorrelated with everything. Do you expect
  it to carry useful class signal? What will a high-C SVM do with a
  feature that has no real signal but some training-set structure by chance?

### 4.4 Preprocessing

SVMs require standardised features: the kernel distance calculation
treats all features symmetrically, so features on different scales
will dominate the kernel. `log_xray_flux` ranges from -9 to -3;
`magnetic_bmax` ranges from 0 to 3500; `radio_burst_sfu` ranges from
0 to 15,000. Without scaling, the large-range features would dominate
every distance calculation.

Apply `StandardScaler` fit on training data only.

In [ ]:
# Listing 6.
# TODO: Preprocessing.
#
# 1. Separate features (X) from the target ('activity_class') (y).
#    Use all 12 columns in FEATURE_COLS.
#
# 2. Split into training and test sets.
#    Use an 80/20 split, random_state=42, stratify=y.
#    Print class counts in y_train and y_test.
#
# 3. Apply StandardScaler: fit on X_train, transform both.
#    Store as X_train_sc and X_test_sc.

# YOUR CODE HERE

---
## 5. Part 1: The Overfitting SVM

### 5.1 What Overfitting Looks Like

An SVM overfits when it draws a decision boundary so tightly around
the training examples that the boundary does not reflect the true
underlying class structure. In feature space, an overfitting RBF SVM
creates many small, island-like decision regions around individual
training points or small clusters. The training accuracy approaches
100% because the model can always find a boundary that separates
the training points. But when a new test point arrives in a region
the model has overspecialised around, it is likely to be misclassified.

Two hyperparameters control RBF SVM flexibility:

- **C** (regularisation parameter): controls the tradeoff between
  maximising the margin and minimising training errors. High C = small
  margin, few support vectors, tight fit, tends to overfit. Low C =
  large margin, many support vectors, smooth decision boundary.

- **gamma**: controls the width of the RBF kernel. Each training point
  influences the decision boundary within a radius of approximately
  1/sqrt(gamma). High gamma = narrow influence, each point becomes
  its own island, extreme overfitting. Low gamma = broad influence,
  smooth boundaries, may underfit.

The interaction between C and gamma is what makes RBF SVM tuning
non-trivial. Both can independently cause overfitting, and their
effects compound: a high-C, high-gamma SVM can memorise 10,000
training examples perfectly while failing on test data.

### 5.2 Demonstrating Overfitting

In [ ]:
# Listing 7.
# TODO: Demonstrate overfitting with a poorly configured SVM.
#
# Train two SVMs:
#
# Model A: deliberately overfitting configuration
#   SVC(kernel='rbf', C=100.0, gamma=1.0, random_state=42)
#   This has very high C (ignores margin) and very high gamma
#   (narrow influence radius).
#
# Model B: default configuration
#   SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
#
# For each model, compute and print:
#   - Training accuracy
#   - Test accuracy
#   - The gap (training - test)
#   - Number of support vectors (clf.n_support_)
#
# Then print the full classification report for both models on the
# test set, using target_names=CLASS_NAMES.
#
# Questions to think about:
#   - How does the number of support vectors differ between the two models?
#     A model that memorises the training data needs more support vectors.
#   - Model A has perfect training accuracy but poor test accuracy.
#     Which classes are most affected by the overfitting?

# YOUR CODE HERE

---
## 6. Part 2: The Role of C and Gamma

### 6.1 The C Parameter

With gamma fixed at a reasonable value, sweep C from very small
(strong regularisation) to very large (weak regularisation) and
record training and test accuracy at each value. This isolates the
effect of C on the bias-variance tradeoff.

In [ ]:
# Listing 8.
# TODO: C sweep with fixed gamma.
#
# 1. Set gamma=0.1 (a moderate value that makes the C effect visible).
#    Loop over C values: [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0, 1000.0]
#    For each C:
#      - Train SVC(kernel='rbf', C=C, gamma=0.1, random_state=42) on X_train_sc
#      - Record training accuracy and test accuracy
#      - Print a row: C, train_acc, test_acc, gap
#
# 2. Plot training accuracy and test accuracy vs log10(C) on the same axes.
#    Use a log scale on the x-axis.
#    Mark the optimal C (highest test accuracy) with a vertical dashed line.
#    Title: 'SVM: Training vs Test Accuracy as C Varies (gamma=0.1)'
#
# 3. Add a second y-axis (or a separate panel below) showing the
#    train-test gap vs C. This makes the overfitting region visually clear.
#
# 4. Describe the shape of the curve:
#    - At very low C: both training and test accuracy are low (underfitting)
#    - At optimal C: test accuracy peaks
#    - At high C: training accuracy continues rising, test accuracy falls (overfitting)

# YOUR CODE HERE

### 6.2 The Gamma Parameter

With C fixed at a moderate value, sweep gamma from very small
(smooth, broad kernel) to very large (narrow, tight kernel). This
isolates the effect of gamma on overfitting.

In [ ]:
# Listing 9.
# TODO: Gamma sweep with fixed C.
#
# 1. Set C=10.0.
#    Loop over gamma values: [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
#    For each gamma:
#      - Train SVC(kernel='rbf', C=10.0, gamma=gamma, random_state=42)
#      - Record training and test accuracy
#      - Print: gamma, train_acc, test_acc, gap
#
# 2. Plot training accuracy and test accuracy vs log10(gamma).
#    Mark the optimal gamma with a vertical dashed line.
#    Title: 'SVM: Training vs Test Accuracy as Gamma Varies (C=10.0)'
#
# 3. Compare the shape to the C sweep:
#    - Does gamma overfit more aggressively than C for the same range
#      of multiplicative change?
#    - At what gamma value does training accuracy first reach ~100%?
#    - What is the test accuracy at that point?

# YOUR CODE HERE

### 6.3 The Joint Effect of C and Gamma

C and gamma do not act independently. A high-C, low-gamma model may
behave differently from a low-C, high-gamma model even if both produce
the same training accuracy. To see the full landscape, create a 2D
heatmap of test accuracy over a grid of C and gamma values.

In [ ]:
# Listing 10.
# TODO: C-gamma heatmap.
#
# 1. Define:
#    C_values     = [0.01, 0.1, 1.0, 10.0, 100.0]
#    gamma_values = [0.001, 0.01, 0.1, 1.0, 10.0]
#
# 2. For every combination of (C, gamma):
#    - Train SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
#    - Record test accuracy
#    Store as a 5x5 matrix (rows=C, cols=gamma).
#
# 3. Create three heatmaps side by side:
#    Left:   Training accuracy heatmap (rows=C, cols=gamma)
#    Centre: Test accuracy heatmap
#    Right:  Train-test gap heatmap
#    Use seaborn.heatmap with annot=True and fmt='.3f'.
#    Use 'YlOrRd' for gap (red = worse) and 'YlGn' for test accuracy.
#
# 4. Which region of the C-gamma space shows the worst overfitting?
#    Which region gives the best test accuracy?
#    Is the best test accuracy region in a corner, or in the interior?

# YOUR CODE HERE

---
## 7. Part 3: Finding the Right Regularisation

### 7.1 Grid Search with Cross-Validation

The heatmap in Section 6.3 was computed on the test set. This is
informative for visualisation but should not be used to select
hyperparameters: evaluating configurations on the test set and picking
the best one is a form of data leakage. The test set is no longer a
fair estimate of generalisation once you have used it for selection.

`GridSearchCV` uses stratified k-fold cross-validation on the training
set to select hyperparameters. Each fold is an independent held-out set
from the training data; the final selected configuration has never seen
the test set during the search.

In [ ]:
# Listing 11.
# TODO: Grid search for the best C and gamma.
#
# 1. Define the parameter grid:
#    param_grid = {
#        'C':     [0.01, 0.1, 1.0, 10.0, 100.0],
#        'gamma': [0.001, 0.01, 0.1, 1.0, 10.0],
#        'kernel': ['rbf'],
#    }
#
# 2. Create StratifiedKFold(n_splits=5, shuffle=True, random_state=42).
#
# 3. Create GridSearchCV:
#    gs = GridSearchCV(
#        estimator=SVC(random_state=42),
#        param_grid=param_grid,
#        cv=cv,
#        scoring='f1_macro',
#        n_jobs=-1,
#        verbose=1,
#        refit=True
#    )
#    Fit on X_train_sc and y_train.
#
# 4. Print gs.best_params_ and gs.best_score_.
#
# 5. Print the top 8 configurations from gs.cv_results_, sorted by
#    mean_test_score descending.
#
# Note: use scoring='f1_macro' rather than accuracy. With three classes
# and a 44:36:20 distribution, accuracy can be high while the eruptive
# class is poorly recalled. Macro F1 gives equal weight to all three classes.

# YOUR CODE HERE

### 7.2 Final Evaluation and Comparison

In [ ]:
# Listing 12.
# TODO: Compare overfitting model vs best regularised model.
#
# 1. Use gs.best_estimator_ to predict on X_test_sc.
#    Compute accuracy, macro-F1, and classification report.
#
# 2. Create a summary comparison table (DataFrame) with three rows:
#    - Overfitting SVM (C=100, gamma=1.0)
#    - Default SVM (C=1.0, gamma='scale')
#    - Best regularised SVM (from grid search)
#    Columns: train_accuracy, test_accuracy, gap, macro_f1, n_support_vectors
#
# 3. Plot side-by-side confusion matrices for all three models.
#    Use display_labels=CLASS_NAMES.
#    Title each panel clearly.
#
# 4. Answer in a markdown cell:
#    - Which model has the smallest train-test gap?
#    - Which model has the highest test accuracy?
#    - Does the best C-gamma combination from the grid search match
#      what you found as optimal in the heatmap of Section 6.3?
#      If not, why might they differ?
#    - What is the eruptive class recall in each model? A missed eruptive
#      event has severe real-world consequences. Did regularisation help?

# YOUR CODE HERE

### 7.3 Reflection Questions

1. The overfitting SVM (C=100, gamma=1.0) achieves ~100% training
   accuracy. How many support vectors does it have compared to the
   well-regularised model? Explain the relationship between the number
   of support vectors and the degree of overfitting.

2. In the C sweep (Section 6.1), you should see that very low C causes
   underfitting (low training AND test accuracy) while high C causes
   overfitting. Where is the crossover point? At the crossover, what
   is the train-test gap, and what does this tell you about the
   irreducible noise in the dataset (the label noise)?

3. The gamma sweep (Section 6.2) typically shows more dramatic
   overfitting than the C sweep for the same range of multiplicative
   change. Why is gamma more dangerous than C in this respect?
   (Hint: think about what gamma = 10 means for the effective radius
   of influence of each training point in the scaled feature space.)

4. The grid search used `scoring='f1_macro'` to select hyperparameters.
   If you had used `scoring='accuracy'` instead, would you expect the
   selected (C, gamma) to be different? Which metric is more appropriate
   for this problem and why?

5. The SVM was trained on all 12 features including the 6 noisy derived
   features. If you removed the 6 noisy features and trained only on
   the 6 informative features, how would you expect the overfitting
   to change? Would the optimal C and gamma shift? How would you
   test this empirically without looking at the test set?

---
## 8. Solution

**Read this section only after attempting the challenge yourself.**

### Step 1: Preprocessing

In [ ]:
# Listing 13.
X = df[FEATURE_COLS].values
y = df['activity_class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Class counts in training set:')
for cls in range(3):
    n = int(np.sum(y_train == cls))
    print(f'  {CLASS_NAMES[cls]:12s}: {n:5d}  ({100*n/len(y_train):.1f}%)')

### Step 2: Demonstrating Overfitting

In [ ]:
# Listing 14.
configs = {
    'Overfitting (C=100, gamma=1.0)': SVC(kernel='rbf', C=100.0, gamma=1.0,  random_state=42),
    'Default    (C=1,   gamma=scale)': SVC(kernel='rbf', C=1.0,   gamma='scale', random_state=42),
}

for name, clf in configs.items():
    clf.fit(X_train_sc, y_train)
    tr = accuracy_score(y_train, clf.predict(X_train_sc))
    te = accuracy_score(y_test,  clf.predict(X_test_sc))
    nsv = sum(clf.n_support_)
    print(f'{name}')
    print(f'  Train acc: {tr:.4f}  Test acc: {te:.4f}  Gap: {tr-te:.4f}  '
          f'Support vectors: {nsv}')
    print()

print('Classification reports:')
for name, clf in configs.items():
    print(f'\n--- {name} ---')
    print(classification_report(y_test, clf.predict(X_test_sc),
                                target_names=CLASS_NAMES))

The overfitting SVM achieves near-perfect training accuracy (it has
memorised the training set) but substantially lower test accuracy.
It also uses far more support vectors: a memorising SVM needs a support
vector near every training example to define its tight local boundaries.
The default SVM uses far fewer support vectors and generalises better.

Notice also how the number of support vectors relates to generalisation:
in the SVM theory, the generalisation bound depends on the ratio of
support vectors to training samples. A model with 90% of training
examples as support vectors has very little margin and is known to
generalise poorly.

### Step 3: C Sweep

In [ ]:
# Listing 15.
C_values = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0, 1000.0]
train_accs_C = []; test_accs_C = []

print(f'{"C":>10}  {"train":>8}  {"test":>8}  {"gap":>8}')
for C in C_values:
    clf = SVC(kernel='rbf', C=C, gamma=0.1, random_state=42)
    clf.fit(X_train_sc, y_train)
    tr = accuracy_score(y_train, clf.predict(X_train_sc))
    te = accuracy_score(y_test,  clf.predict(X_test_sc))
    train_accs_C.append(tr); test_accs_C.append(te)
    print(f'{C:>10.3f}  {tr:>8.4f}  {te:>8.4f}  {tr-te:>8.4f}')

best_C_idx = int(np.argmax(test_accs_C))
best_C_val = C_values[best_C_idx]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
log_C = np.log10(C_values)

ax1.plot(log_C, train_accs_C, 'o-', color='steelblue', linewidth=2, label='Training accuracy')
ax1.plot(log_C, test_accs_C,  's-', color='firebrick', linewidth=2, label='Test accuracy')
ax1.axvline(np.log10(best_C_val), color='grey', linestyle='--', linewidth=1.5,
            label=f'Best C={best_C_val}')
ax1.set_ylabel('Accuracy', fontsize=11)
ax1.set_title('SVM: Effect of C on Training vs Test Accuracy (gamma=0.1)', fontsize=12)
ax1.legend(fontsize=10)

gaps_C = [tr - te for tr, te in zip(train_accs_C, test_accs_C)]
ax2.bar(log_C, gaps_C, width=0.2, color='darkorange', alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axvline(np.log10(best_C_val), color='grey', linestyle='--', linewidth=1.5)
ax2.set_xlabel('log10(C)', fontsize=11)
ax2.set_ylabel('Train - Test Gap', fontsize=11)
ax2.set_title('Overfitting Gap vs C', fontsize=12)

plt.tight_layout()
plt.show()

The C sweep shows the classic bias-variance tradeoff curve:

At very low C (strong regularisation), the model underfits: the
margin constraint is so tight that the model cannot fit even the
genuine structure in the data, and both training and test accuracy
are low.

As C increases toward the optimal range, training accuracy rises and
test accuracy rises with it: the model is capturing genuine class
structure without over-specialising. The train-test gap is small.

At high C (weak regularisation), training accuracy approaches 100%
but test accuracy falls. The model has memorised the training examples,
including the noisy label-flipped ones and the borderline cases where
the classes genuinely overlap. The train-test gap grows: this is the
overfitting signature.

### Step 4: Gamma Sweep

In [ ]:
# Listing 16.
gamma_values = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
train_accs_g = []; test_accs_g = []

print(f'{"gamma":>10}  {"train":>8}  {"test":>8}  {"gap":>8}')
for g in gamma_values:
    clf = SVC(kernel='rbf', C=10.0, gamma=g, random_state=42)
    clf.fit(X_train_sc, y_train)
    tr = accuracy_score(y_train, clf.predict(X_train_sc))
    te = accuracy_score(y_test,  clf.predict(X_test_sc))
    train_accs_g.append(tr); test_accs_g.append(te)
    print(f'{g:>10.4f}  {tr:>8.4f}  {te:>8.4f}  {tr-te:>8.4f}')

best_g_idx = int(np.argmax(test_accs_g))
best_g_val = gamma_values[best_g_idx]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
log_g = np.log10(gamma_values)

ax1.plot(log_g, train_accs_g, 'o-', color='steelblue', linewidth=2, label='Training accuracy')
ax1.plot(log_g, test_accs_g,  's-', color='firebrick', linewidth=2, label='Test accuracy')
ax1.axvline(np.log10(best_g_val), color='grey', linestyle='--', linewidth=1.5,
            label=f'Best gamma={best_g_val}')
ax1.set_ylabel('Accuracy', fontsize=11)
ax1.set_title('SVM: Effect of Gamma on Training vs Test Accuracy (C=10.0)', fontsize=12)
ax1.legend(fontsize=10)

gaps_g = [tr - te for tr, te in zip(train_accs_g, test_accs_g)]
ax2.bar(log_g, gaps_g, width=0.15, color='darkorange', alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axvline(np.log10(best_g_val), color='grey', linestyle='--', linewidth=1.5)
ax2.set_xlabel('log10(gamma)', fontsize=11)
ax2.set_ylabel('Train - Test Gap', fontsize=11)
ax2.set_title('Overfitting Gap vs Gamma', fontsize=12)

plt.tight_layout()
plt.show()

The gamma sweep is more dramatic than the C sweep. At low gamma, the
kernel has a very broad influence radius: each training point influences
the decision boundary far away from itself, producing a smooth, global
boundary. This can underfit if gamma is too low.

As gamma increases toward the optimal, test accuracy rises. But beyond
the optimal, training accuracy shoots toward 100% very quickly while
test accuracy falls steeply. This is because high gamma makes each
training point an island: the kernel only activates for test points
that are extremely close to a training example, so the model can only
classify correctly in the immediate neighbourhood of known training data.

At gamma=1.0 with C=10, the model achieves ~100% training accuracy
but drops to ~91% test accuracy, an 9 percentage point gap. At gamma=5.0,
the model has degenerated further. The lesson: **gamma is the primary
source of overfitting in RBF SVMs.** Controlling gamma matters more
than controlling C for avoiding memorisation.

### Step 5: C-Gamma Heatmap (slow, 30 seconds ish)

In [ ]:
# Listing 17.
C_grid     = [0.01, 0.1, 1.0, 10.0, 100.0]
gamma_grid = [0.001, 0.01, 0.1, 1.0, 10.0]

train_mat = np.zeros((len(C_grid), len(gamma_grid)))
test_mat  = np.zeros((len(C_grid), len(gamma_grid)))

for i, C in enumerate(C_grid):
    for j, g in enumerate(gamma_grid):
        clf = SVC(kernel='rbf', C=C, gamma=g, random_state=42)
        clf.fit(X_train_sc, y_train)
        train_mat[i, j] = accuracy_score(y_train, clf.predict(X_train_sc))
        test_mat[i, j]  = accuracy_score(y_test,  clf.predict(X_test_sc))

gap_mat = train_mat - test_mat

C_labels = [str(c) for c in C_grid]
g_labels = [str(g) for g in gamma_grid]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, data, title, cmap in zip(
    axes,
    [train_mat, test_mat, gap_mat],
    ['Training Accuracy', 'Test Accuracy', 'Overfitting Gap (Train - Test)'],
    ['YlGn', 'YlGn', 'YlOrRd']
):
    sns.heatmap(data, annot=True, fmt='.3f', cmap=cmap,
                xticklabels=g_labels, yticklabels=C_labels,
                ax=ax, vmin=0 if 'Gap' in title else 0.7,
                vmax=0.3 if 'Gap' in title else 1.0)
    ax.set_xlabel('gamma', fontsize=11)
    ax.set_ylabel('C', fontsize=11)
    ax.set_title(title, fontsize=12)

plt.suptitle('SVM C-Gamma Grid: SolarObs Dataset', fontsize=13)
plt.tight_layout()
plt.show()

The heatmap makes the overfitting landscape vivid:

- Top-right corner (high C, high gamma): training accuracy = 1.000,
  but test accuracy may be as low as 0.4-0.7. The gap is extreme.
- Bottom-left corner (low C, low gamma): both training and test accuracy
  are lower but similar to each other. The model underfits.
- The highest test accuracy occurs somewhere in the interior of the
  grid, typically around C=1-10 and gamma=0.01-0.1. This is the
  well-regularised region.

The gap heatmap (right panel) is the clearest diagnostic: it shows
exactly where overfitting occurs. A well-tuned model lives in the
dark region of the gap heatmap, not the bright region.

### Step 6: Grid Search

In [ ]:
# Listing 18.
param_grid = {
    'C':      [0.01, 0.1, 1.0, 10.0, 100.0],
    'gamma':  [0.001, 0.01, 0.1, 1.0, 10.0],
    'kernel': ['rbf'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=0,
    refit=True
)
gs.fit(X_train_sc, y_train)

print(f'Best parameters: {gs.best_params_}')
print(f'Best CV macro-F1: {gs.best_score_:.4f}')

results_df = (pd.DataFrame(gs.cv_results_)
              [['param_C', 'param_gamma', 'mean_test_score', 'std_test_score']]
              .sort_values('mean_test_score', ascending=False)
              .head(8))
print('\nTop 8 configurations by CV macro-F1:')
print(results_df.to_string(index=False))

The grid search should find the best (C, gamma) in the well-regularised
region: typically C=1-10 and gamma=0.01-0.1. This is consistent with
the test accuracy heatmap from Section 6.3, confirming that
cross-validation on the training set successfully identifies the
configurations that generalise well.

The difference between the grid search result and the heatmap best
is usually small (within 1-2 percentage points) but the grid search
result is more trustworthy: it was never exposed to the test set.

### Step 7: Final Comparison

In [ ]:
# Listing 19.
# Fit all three comparison models
clf_overfit  = SVC(kernel='rbf', C=100.0,  gamma=1.0,     random_state=42)
clf_default  = SVC(kernel='rbf', C=1.0,    gamma='scale', random_state=42)
clf_best     = gs.best_estimator_

models = {
    f'Overfitting (C=100, g=1.0)':     clf_overfit,
    f'Default (C=1, g=scale)':         clf_default,
    f'Tuned (C={gs.best_params_["C"]}, g={gs.best_params_["gamma"]})': clf_best,
}

for name, clf in models.items():
    if clf != clf_best:
        clf.fit(X_train_sc, y_train)

rows = []
for name, clf in models.items():
    tr = accuracy_score(y_train, clf.predict(X_train_sc))
    te = accuracy_score(y_test,  clf.predict(X_test_sc))
    f1 = f1_score(y_test, clf.predict(X_test_sc), average='macro')
    nsv = sum(clf.n_support_)
    rows.append({'Model': name, 'Train Acc': tr, 'Test Acc': te,
                 'Gap': tr-te, 'Macro-F1': f1, 'N Support Vectors': nsv})

comparison_df = pd.DataFrame(rows)
print('Model comparison:')
print(comparison_df.round(4).to_string(index=False))

In [ ]:
# Listing 20.
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, clf) in zip(axes, models.items()):
    yp = clf.predict(X_test_sc)
    ConfusionMatrixDisplay.from_predictions(
        y_test, yp, display_labels=CLASS_NAMES,
        cmap='Blues', ax=ax, colorbar=False
    )
    te = accuracy_score(y_test, yp)
    tr = accuracy_score(y_train, clf.predict(X_train_sc))
    ax.set_title(f'{name}\ntrain={tr:.3f}  test={te:.3f}', fontsize=9)

plt.suptitle('Confusion Matrix Comparison: SolarObs', fontsize=13)
plt.tight_layout()
plt.show()

**Summary:**

The overfitting SVM memorises the training set but fails on test data.
The confusion matrix shows it misclassifies many active events as
either quiet or eruptive: without a well-defined margin, the decision
boundary in the active region is irregular and non-generalising.

The well-regularised SVM (from the grid search) achieves the highest
test accuracy and lowest train-test gap. Its confusion matrix shows
the expected pattern: most errors occur at the quiet/active and
active/eruptive boundaries, which reflect genuine class overlap rather
than model failure.

The key numerical lesson: the overfitting SVM has training accuracy
approaching 100% with a test accuracy that may be 10-20 percentage
points lower. The tuned model has a small gap (1-4 percentage points)
and higher absolute test accuracy. This is the bias-variance tradeoff
made visible: the overfitting model has low bias (it fits every
training example) but high variance (its predictions are unstable
for new examples). The tuned model accepts slightly higher training
error in exchange for lower test error.

---
## 9. References

1. Cortes, C. and Vapnik, V. (1995). Support-vector networks.
   *Machine Learning*, 20(3), 273-297.
   https://doi.org/10.1007/BF00994018
   Original SVM paper introducing the maximum-margin classifier and
   the soft-margin formulation controlled by the C parameter.

2. Scholkopf, B. and Smola, A. (2002). *Learning with Kernels*.
   MIT Press.
   Comprehensive treatment of kernel methods including the RBF kernel,
   the role of gamma, and the theory of generalisation bounds for SVMs.

3. Hsu, C.W., Chang, C.C., and Lin, C.J. (2003). A practical guide
   to support vector classification. Technical Report, National Taiwan
   University. https://www.csie.ntu.edu.tw/~cjlin/papers/guide/guide.pdf
   Practical advice on choosing C and gamma for RBF SVMs, including
   the log-scale grid search strategy used in this challenge.

4. Bobra, M.G. and Couvidat, S. (2015). Solar flare prediction using
   SDO/HMI vector magnetic field data with a machine-learning algorithm.
   *The Astrophysical Journal*, 798(2), 135.
   https://doi.org/10.1088/0004-637X/798/2/135
   Application of SVMs to real solar flare prediction using SDO/HMI
   magnetogram features. The feature set in this challenge is partially
   inspired by the SHARP parameters used in this paper.

5. Huang, X. et al. (2018). Solar flare prediction using photospheric
   magnetic field data with deep neural networks.
   *The Astrophysical Journal Supplement Series*, 236(1), 1.
   https://doi.org/10.3847/1538-4365/aab9a8
   Comparison of SVM and deep learning approaches on solar flare
   prediction, providing context for why SVM regularisation matters
   in this domain.

6. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html